# Metropolis-Hastings sampling

Joshua French

An interactive version of this content is available as a Colab notebook, which can be accessed by scanning or clicking the QR code below.

<a href="https://colab.research.google.com/github/jfrench/BayesianStatistics/blob/master/ch03e-mh.ipynb"> <img src="https://raw.githubusercontent.com/jfrench/BayesianStatistics/refs/heads/master/qr_codes/mh.svg" width="300" height="300"> </a>

------------------------------------------------------------------------

These notes cover the Metropolis and Metropolis-Hastings algorithms and methods for assessing MCMC convergence. They draw on BDA3 Chapter 11 and Chapters 6 and 7 of *A First Course in Bayesian Statistical Methods* (Hoff).

------------------------------------------------------------------------

We will need certain packages to be available for this notebook. The code below will check if they are installed and install them if necessary.

In [1]:
if (!require("remotes", quietly = TRUE)) {
  install.packages("remotes", repos = "https://cran.rstudio.com/")
}
if (!require("bayesutils", quietly = TRUE)) {
  remotes::install_github("jfrench/bayesutils")
}
if (!require("mvtnorm", quietly = TRUE)) {
  install.packages("mvtnorm", repos = "https://cran.rstudio.com/")
}
if (!require("posterior", quietly = TRUE)) {
  install.packages("posterior", repos = "https://cran.rstudio.com/")
}
if (!require("hexbin", quietly = TRUE)) {
  install.packages("hexbin", repos = "https://cran.rstudio.com/")
}
if (!require("bayesplot", quietly = TRUE)) {
  install.packages("bayesplot", repos = "https://cran.rstudio.com/")
}

------------------------------------------------------------------------

## Attach necessary R packages

In [2]:
library(bayesutils)
library(mvtnorm)
library(posterior)
library(bayesplot)

------------------------------------------------------------------------

## The Metropolis and Metropolis-Hastings algorithms

- The **Metropolis-Hastings algorithm** is a general term for a family of Markov chain simulation methods useful for sampling from Bayesian posterior distributions.
- The Gibbs sampler is a special case of the Metropolis-Hastings algorithm.

------------------------------------------------------------------------

### Metropolis-Hastings algorithm

1.  Choose an initial value, $\theta^{(0)}$, for which $p(\theta^{(0)} \mid y) > 0$.
2.  For $t=1,2,\ldots$:

------------------------------------------------------------------------

1.  Sample a proposal $\theta^*$ from a **proposal distribution** (or jumping distribution) at time $t$, $$
       \theta^* \sim j_t(\theta^* \mid \theta^{(t-1)}).
       $$
2.  Calculate the acceptance ratio $$
       r = \frac{p(\theta^* \mid y)/j_t(\theta^* \mid \theta^{(t-1)})}{p(\theta^{(t-1)} \mid y)/j_t(\theta^{(t-1)} \mid \theta^*)}=\frac{p(\theta^* \mid y)j_t(\theta^{(t-1)} \mid \theta^*)}{p(\theta^{(t-1)} \mid y)j_t(\theta^* \mid \theta^{(t-1)})}.
       $$

------------------------------------------------------------------------

1.  Set $$
       \theta^{(t)} =
       \begin{cases}
       \theta^*, & \text{with probability } \min(r,1),\\
       \theta^{(t-1)}, & \text{otherwise}.
       \end{cases}
       $$

------------------------------------------------------------------------

The **Metropolis algorithm** assumes a symmetric proposal distribution.

- A symmetric proposal distribution means that $j_t(\theta^* \mid \theta^{(t-1)}) = j_t(\theta^{(t-1)} \mid \theta^*)$.
- The acceptance ratio simplifies to $$
  r = \frac{p(\theta^* \mid y)}{p(\theta^{(t-1)} \mid y)}.
  $$

------------------------------------------------------------------------

Some intuition (based on the Metropolis algorithm):

- If we propose a value with greater target density, then we move to that value with probability one.
- If we propose a value with smaller target density, we will still move to that value with probability $r$.

The Markov chain must sometimes move to less likely regions or it would not explore the entire target distribution.

------------------------------------------------------------------------

### Computational note

- To avoid numerical overflow or underflow issues, we typically implement the acceptance rule on the natural log scale.
- We calculate $\log(r)$ and accept $\theta^*$ when $$
  \log(u) \leq \min\{\log(r),0\},
  $$ where $u \sim \text{Uniform}(0,1)$.

------------------------------------------------------------------------

## Tips for choosing a proposal distribution

- It should be easy to sample from $j_t(\theta^* \mid \theta)$ for all $\theta$.
- It should be easy to compute the acceptance ratio $r$.
- Each proposal value should go a reasonable distance in the parameter space.
- The proposed values should be accepted between 20–50% of the time.
  - For an independence sampler, the acceptance rate should be as high as possible.

------------------------------------------------------------------------

## Types of the M-H algorithm

**Random walk**

- The proposal is symmetric and depends only on the distance between parameters, i.e., $j_t(\theta^* \mid \theta^{(t-1)}) = j_t\left(\left|\theta^* - \theta^{(t-1)}\right|\right)$.
- E.g., $\theta^* \mid \theta^{(t-1)} \sim N(\theta^{(t-1)},\sigma^2)$.
- This is a special case of the Metropolis algorithm.

------------------------------------------------------------------------

**Independence Chain**

- The proposal distribution does not depend on the past.
- i.e., $j_t(\theta^* \mid \theta^{(t-1)}) = j(\theta^*)$ for a fixed proposal density $j$.
- The proposal distribution should resemble the target distribution and cover the tails of the target distribution.
- A simple trick is to use the prior as the proposal distribution. In that case, $$
  r
  = \frac{p(y \mid \theta^*)p(\theta^*)/p(\theta^*)}{p(y \mid \theta^{(t-1)})p(\theta^{(t-1)})/p(\theta^{(t-1)})}
  = \frac{p(y \mid \theta^*)}{p(y \mid \theta^{(t-1)})}.
  $$

------------------------------------------------------------------------

**Componentwise Metropolis-Hastings**

- Divide the parameter vector $\theta$ into subvectors and update sequentially using Metropolis-Hastings updates to draw samples from each subcomponent.
- This gets a lot more complicated.

------------------------------------------------------------------------

## Example: Random walk Metropolis

Use a random walk Metropolis algorithm to sample from the posterior distribution in the following setting:

**Data distribution:** $y \mid \theta \sim N(\theta,I)$, where $\theta$ is a vector of length 2, $I$ is the $2\times2$ identity matrix, and the observed $y=(0,0)$.

**Prior distribution:** $p(\theta) \propto 1$ (i.e., an improper prior).

**Proposal distribution:** $\theta^* \mid \theta^{(t-1)} \sim N(\theta^{(t-1)},cI)$, where $c$ is a constant chosen to scale the size of the proposal distribution.

------------------------------------------------------------------------

**Acceptance ratio:**

------------------------------------------------------------------------

We implement a Metropolis-Hastings function with the following parameters:

- `B`: number of iterations
- `start`: vector of starting values
- `const`: variance of proposal distribution
- `chain_id`: a unique id for each chain

------------------------------------------------------------------------

In [3]:
mh <- function(B, start, const, chain_id = factor(runif(1))) {
  Imat <- diag(length(start)) # Identity matrix
  theta <- matrix(0, nrow = B + 1, ncol = length(start)) # store iterations
  theta[1, ] <- start # first row is the starting values
  accepted <- 0
  for (i in seq_len(B) + 1) {
    # sample from proposal distribution
    theta_star <- rmvnorm(1, mean = theta[i - 1, ], sigma = const * Imat)
    # special case: acceptance ratio is the ratio of likelihoods.
    r <- dmvnorm(c(0, 0), mean = theta_star, sigma = Imat) /
         dmvnorm(c(0, 0), mean = theta[i - 1, ], sigma = Imat)
    if (runif(1) <= min(r, 1)) { # Accept or reject the proposed value
      theta[i, ] <- theta_star
      accepted <- accepted + 1
    } else {
      theta[i, ] <- theta[i - 1, ]
    }
  }
  # return in format for posterior package
list(
  draws = posterior::as_draws_df(
    data.frame(
      .chain = chain_id,
      .iteration = seq_len(B + 1),
      theta1 = theta[, 1],
      theta2 = theta[, 2]
    )
  ),
  acceptance_rate = accepted / B
)
}

------------------------------------------------------------------------

In [4]:
set.seed(1234) # Reproducibility
B <- 1000 # iterations
# Proposal standard deviation = 0.2, so proposal variance = 0.2^2
const <- 0.2^2
# run several chains
mh1 <- mh(B, c(0, 0), const)
mh2 <- mh(B, c(-3, 3), const)
mh3 <- mh(B, c(-3, -3), const)
mh4 <- mh(B, c(3, -3), const)
mh5 <- mh(B, c(3, 3), const)

------------------------------------------------------------------------

In [5]:
# starting values
start_x <- c(-3, -3, 0, 3, 3)
start_y <- c(-3, 3, 0, -3, 3)
# plot first 50 steps of M-H chains
plot(c(0, 0), xlim = c(-4, 4), ylim = c(-4, 4), type = "n",
  xlab = expression(theta[1]), ylab = expression(theta[2]))
lines(mh1$draws$theta1[1:51], mh1$draws$theta2[1:51])
lines(mh2$draws$theta1[1:51], mh2$draws$theta2[1:51])
lines(mh3$draws$theta1[1:51], mh3$draws$theta2[1:51])
lines(mh4$draws$theta1[1:51], mh4$draws$theta2[1:51])
lines(mh5$draws$theta1[1:51], mh5$draws$theta2[1:51])
points(start_x, start_y, pch = 20)
title("First 50 iterations of M-H chains")

------------------------------------------------------------------------

------------------------------------------------------------------------

In [7]:
# Plot all steps of M-H chains
plot(c(0, 0), xlim = c(-4, 4), ylim = c(-4, 4), type = "n",
  xlab = expression(theta[1]), ylab = expression(theta[2]))
lines(mh1$draws$theta1, mh1$draws$theta2)
lines(mh2$draws$theta1, mh2$draws$theta2)
lines(mh3$draws$theta1, mh3$draws$theta2)
lines(mh4$draws$theta1, mh4$draws$theta2)
lines(mh5$draws$theta1, mh5$draws$theta2)
points(start_x, start_y, pch = 20)
title("First 1000 iterations of M-H chains")

------------------------------------------------------------------------

------------------------------------------------------------------------

In [9]:
# Plot second half of posterior sample
keep <- (B / 2 + 1):(B + 1)
plot(mh1$draws$theta1[keep], mh1$draws$theta2[keep],
     pch = ".", col = "brown",
  xlim = c(-4, 4), ylim = c(-4, 4),
  xlab = expression(theta[1]), ylab = expression(theta[2])
)
points(mh2$draws$theta1[keep], mh2$draws$theta2[keep], pch = ".", col = "blue")
points(mh3$draws$theta1[keep], mh3$draws$theta2[keep], pch = ".", col = "orange")
points(mh4$draws$theta1[keep], mh4$draws$theta2[keep], pch = ".", col = "purple")
points(mh5$draws$theta1[keep], mh5$draws$theta2[keep], pch = ".", col = "cyan")

------------------------------------------------------------------------

------------------------------------------------------------------------

We repeat with a larger proposal variance.

In [11]:
const <- 1
mh1 <- mh(B, c(0, 0), 1)
mh2 <- mh(B, c(-3, 3), 1)
mh3 <- mh(B, c(-3, -3), 1)
mh4 <- mh(B, c(3, -3), 1)
mh5 <- mh(B, c(3, 3), 1)

------------------------------------------------------------------------

In [12]:
# starting values
start_x <- c(-3, -3, 0, 3, 3)
start_y <- c(-3, 3, 0, -3, 3)
# plot first 50 steps of M-H chains
plot(c(0, 0), xlim = c(-4, 4), ylim = c(-4, 4), type = "n",
  xlab = expression(theta[1]), ylab = expression(theta[2]))
lines(mh1$draws$theta1[1:51], mh1$draws$theta2[1:51])
lines(mh2$draws$theta1[1:51], mh2$draws$theta2[1:51])
lines(mh3$draws$theta1[1:51], mh3$draws$theta2[1:51])
lines(mh4$draws$theta1[1:51], mh4$draws$theta2[1:51])
lines(mh5$draws$theta1[1:51], mh5$draws$theta2[1:51])
points(start_x, start_y, pch = 20)
title("First 50 iterations of M-H chains")

------------------------------------------------------------------------

------------------------------------------------------------------------

In [14]:
# Plot all steps of M-H chains
plot(c(0, 0), xlim = c(-4, 4), ylim = c(-4, 4), type = "n",
  xlab = expression(theta[1]), ylab = expression(theta[2]))
lines(mh1$draws$theta1, mh1$draws$theta2)
lines(mh2$draws$theta1, mh2$draws$theta2)
lines(mh3$draws$theta1, mh3$draws$theta2)
lines(mh4$draws$theta1, mh4$draws$theta2)
lines(mh5$draws$theta1, mh5$draws$theta2)
points(start_x, start_y, pch = 20)
title("First 1000 iterations of M-H chains")

------------------------------------------------------------------------

------------------------------------------------------------------------

In [16]:
# Plot second half of posterior sample
keep <- (B / 2 + 1):(B + 1)
plot(mh1$draws$theta1[keep], mh1$draws$theta2[keep],
     pch = ".", col = "brown",
  xlim = c(-4, 4), ylim = c(-4, 4),
  xlab = expression(theta[1]), ylab = expression(theta[2])
)
points(mh2$draws$theta1[keep], mh2$draws$theta2[keep], pch = ".", col = "blue")
points(mh3$draws$theta1[keep], mh3$draws$theta2[keep], pch = ".", col = "orange")
points(mh4$draws$theta1[keep], mh4$draws$theta2[keep], pch = ".", col = "purple")
points(mh5$draws$theta1[keep], mh5$draws$theta2[keep], pch = ".", col = "cyan")

------------------------------------------------------------------------

------------------------------------------------------------------------

# Assessing MCMC convergence

------------------------------------------------------------------------

MCMC convergence checks should be performed to make sure that our MCMC chains have converged and adequately explored the target distribution.

**Warmup**

It is common to discard the beginning of an MCMC chain to weaken the influence of the starting value.

- The discarded period is known as the **burn-in** or **warmup**.
- A conservative warmup rule is to discard the first half of the MCMC samples.

------------------------------------------------------------------------

## Multiple chains

- Run multiple chains from dispersed starting values to assess convergence.
- Converged chains should sample from the same target distribution.
- Dispersed starting values help reveal chains stuck in different regions.
- Use multiple numerical and visual diagnostics to assess convergence.

Our random-walk Metropolis example uses five chains with different starting values.

------------------------------------------------------------------------

## Trace plots

A **trace plot** displays sampled parameter values against iteration number.

Trace plots can reveal:

- Persistent trends associated with starting values.
- Chains exploring different regions of the parameter space.
- Long periods during which a chain appears to be stuck.
- Strong autocorrelation or poor mixing.

The sampled values should exhibit random variables without long stretches of getting stuck.

The chains should explore the same parts of the posterior distribution.

------------------------------------------------------------------------

------------------------------------------------------------------------

### Rank plots

A **rank plot** compares the ranks of draws from different chains after pooling the chains and helps determine whether different chains are exploring the same distribution.

- If the chains are sampling from the same distribution, draws from each chain should have approximately uniform ranks.
- Similar rank distributions across chains indicate that the chains are exploring the target distribution similarly.
- Rank plots can reveal differences between chains that are difficult to see in conventional trace plots.

------------------------------------------------------------------------

## Interpreting rank plots

- Approximately flat histograms: Each chain visits the low, middle, and high regions of the pooled draws at roughly the same frequency. This is what we want.
- Histograms concentrated toward low or high ranks: Some chains tend to produce smaller or larger values than others.
- U-shaped or hump-shaped histograms: Chains may differ in their dispersion or tail exploration.
- Substantial differences among chains: Evidence that the chains have not mixed adequately.

------------------------------------------------------------------------

## $\hat{R}$

The **R-hat** diagnostic, written $\hat{R}$, compares variation within chains with variation across chains.

- Modern $\hat{R}$ is rank-normalized and based on split chains.
- Values close to 1 indicate that the chains are behaving similarly.
- As a practical warning threshold, values larger than about 1.01 should be investigated.
- $\hat{R}$ does not prove convergence; it is one diagnostic among several.

------------------------------------------------------------------------

## Effective sample size

The **effective sample size (ESS)** estimates how much information the correlated MCMC draws contain relative to independent draws.

- $B$ correlated MCMC draws generally contain less information than $B$ independent draws.
- **Bulk ESS** assesses sampling efficiency for the central part of the posterior distribution.
- **Tail ESS** is particularly relevant for posterior tails and quantities such as interval endpoints and tail probabilities.

------------------------------------------------------------------------

- We want larger ESS (at least a few hundred, but maybe more).
- More MCMC draws does not guarantee a large effective sample size.

------------------------------------------------------------------------

## Monte Carlo standard error

The **Monte Carlo standard error (MCSE)** quantifies the numerical uncertainty caused by approximating a posterior with a finite MCMC sample.

For example, if a posterior mean is reported to two decimal places, its MCSE should be small relative to 0.01.

------------------------------------------------------------------------

- MCSE is distinct from posterior uncertainty: posterior standard deviation describes uncertainty in the parameter, whereas MCSE describes numerical uncertainty in our estimate of a posterior summary.
- MCSE should be interpreted relative to the precision with which we intend to report a quantity.
- MCSE is most useful after the chains have adequately mixed and the ESS is sufficiently large.

------------------------------------------------------------------------

## Autocorrelation

**Autocorrelation** measures dependence between draws separated by different numbers of iterations.

- Strong autocorrelation means that successive draws provide less new information.
- Slowly decaying autocorrelation indicates inefficient sampling.
- High autocorrelation generally leads to a smaller effective sample size.

We want autocorrelation plots to drop off more quickly, in general.

------------------------------------------------------------------------

## MH-specific tuning diagnostics

For a Metropolis-Hastings sampler, the **acceptance rate** and **autocorrelation** are useful for assessing proposal efficiency.

- A very small proposal variance can lead to high acceptance but highly correlated draws because the chain moves only short distances.
- A very large proposal variance can lead to low acceptance because proposed values frequently land in low-density regions.
- There is no universal acceptance rate that guarantees good sampling. Acceptance rate should be considered together with ESS and autocorrelation.

------------------------------------------------------------------------

- Comparing the two proposal scales illustrates how proposal choice affects MCMC performance.
- Even with the same number of iterations, the chains may have different:
  - Acceptance rates
  - Autocorrelation
  - Effective sample sizes
- The number of MCMC draws does not necessarily reflect the amount of information obtained.

------------------------------------------------------------------------

## Summary of the diagnostic workflow

1.  Run multiple chains from dispersed starting values.
2.  Use trace and rank plots to look for mixing problems.
3.  Check $\hat{R}$ for agreement among chains.
4.  Check bulk and tail ESS for sampling efficiency.
5.  Check MCSE relative to the desired reporting precision.
6.  Examine autocorrelation when ESS is lower than desired.
7.  For M-H, also examine acceptance rate and proposal tuning.

These diagnostics are complementary. No single statistic or plot establishes that an MCMC analysis is adequate.

------------------------------------------------------------------------

### Modern MCMC diagnostics in R

- The `posterior` package provides numerical diagnostics.
- The `bayesplot` package provides visual diagnostics.

The code below converts the five Metropolis-Hastings chains to a `draws_df` object so that the diagnostics can be applied to all chains simultaneously.

In [19]:
draws <- bind_draws(mh1$draws, mh2$draws, mh3$draws,
  mh4$draws, mh5$draws, along = "chain")

------------------------------------------------------------------------

The main numerical diagnostics can be obtained together:

In [20]:
summarize_draws(draws, default_summary_measures())

------------------------------------------------------------------------

In [21]:
summarize_draws(
  draws,
  default_convergence_measures(),
  default_mcse_measures()
)

The output includes posterior summaries together with $\hat{R}$, bulk ESS, tail ESS, and Monte Carlo standard errors.

------------------------------------------------------------------------

A rank plot provides a sensitive visual check of the mixing of the chains:

In [22]:
mcmc_rank_hist(draws, pars = c("theta1", "theta2"))

------------------------------------------------------------------------

A rank ECDF plot provides another rank-based diagnostic and makes systematic deviations from the expected uniform ranks easier to see.

We want the ECDF curves to be inside the confidence bands at a 45-degree angle.

Using 4 or 6 chains makes this plot easier to construct (and you can remove the K = 100).

------------------------------------------------------------------------

In [23]:
# computationally expensive
# mcmc_rank_ecdf(draws, pars = c("theta1", "theta2"), K = 100, prob = 0.99)

------------------------------------------------------------------------

Trace plots remain useful as a complementary visual diagnostic:

In [24]:
mcmc_trace(draws, pars = c("theta1", "theta2"))

------------------------------------------------------------------------

Autocorrelation can be examined directly:

In [25]:
mcmc_acf(draws, pars = c("theta1", "theta2"))

------------------------------------------------------------------------

## Individual diagnostics

In [26]:
summarize_draws(draws, rhat)

In [27]:
summarize_draws(draws, ess_bulk)

In [28]:
summarize_draws(draws, ess_tail)

------------------------------------------------------------------------

In [29]:
summarize_draws(draws, mcse_mean)

------------------------------------------------------------------------

### Example: M-H independence chain for Binomial proportion

Implement the Metropolis-Hastings algorithm with an independence sampler to estimate the posterior for $\theta$, the proportion of female placenta previa births, for the placenta previa data introduced in Chapter 2.

- **Data distribution:** $y \mid \theta \sim \text{Binomial}(980,\theta)$ with observed $y=437$.
- **Prior distribution:** $\theta \sim \text{Uniform}(0,1)$.
- **Proposal distribution:** $\theta^* \sim \text{Beta}(\alpha,\beta)$.

------------------------------------------------------------------------

Because the proposal does not depend on the current state, the Metropolis-Hastings acceptance ratio is

$$
r =
\frac{p(y\mid\theta^*)p(\theta^*)j(\theta^{(t-1)})}
{p(y\mid\theta^{(t-1)})p(\theta^{(t-1)})j(\theta^*)}.
$$

Since the prior is uniform, the prior densities cancel and

$$
r =
\frac{p(y\mid\theta^*)j(\theta^{(t-1)})}
{p(y\mid\theta^{(t-1)})j(\theta^*)}.
$$

------------------------------------------------------------------------

## Implementing the independence sampler

We implement the acceptance rule on the log scale to improve numerical stability.

In [30]:
mh <- function(B, start, jump_parm, chain_id) {

  alpha <- jump_parm[1]
  beta <- jump_parm[2]

  theta <- numeric(B + 1)
  theta[1] <- start

  accepted <- 0

  for (i in 2:(B + 1)) {

    # Generate proposal independently of the current state
    theta_star <- rbeta(1, alpha, beta)

    # Log acceptance ratio
    logr <- dbinom(437, size = 980, prob = theta_star,
                  log = TRUE) -
            dbinom(437, size = 980, prob = theta[i - 1],
                  log = TRUE) +
            dbeta(theta[i - 1], alpha, beta, log = TRUE) -
            dbeta(theta_star, alpha, beta, log = TRUE)

    # Accept or reject
    if (log(runif(1)) <= min(logr, 0)) {
      theta[i] <- theta_star
      accepted <- accepted + 1
    } else {
      theta[i] <- theta[i - 1]
    }
  }

  list(
    draws = posterior::as_draws_df(
      data.frame(
        .chain = chain_id,
        .iteration = seq_len(B + 1),
        theta = theta
      )
    ),
    acceptance_rate = accepted / B
  )
}

------------------------------------------------------------------------

## Running multiple chains

We run four chains from dispersed starting values and use a $\text{Beta}(1,1)$ proposal distribution.

In [31]:
set.seed(77)
B <- 10000
jpar <- c(1, 1)
mh1 <- mh(B, 0.01, jpar, chain_id = 1)
mh2 <- mh(B, 0.25, jpar, chain_id = 2)
mh3 <- mh(B, 0.75, jpar, chain_id = 3)
mh4 <- mh(B, 0.95, jpar, chain_id = 4)

------------------------------------------------------------------------

In [32]:
# Combine chains
draws_all <- bind_draws(mh1$draws, mh2$draws, mh3$draws,
                        mh4$draws, along = "chain")
# Discard the initial state and first half of transitions
draws <- subset_draws(draws_all, iteration = 5002:10001)

The retained sample contains 5,000 draws from each of the four chains.

------------------------------------------------------------------------

## Numerical convergence diagnostics

We assess convergence and sampling efficiency using rank-normalized $\hat R$, bulk ESS, tail ESS, and Monte Carlo standard errors.

In [33]:
summarize_draws(draws)

------------------------------------------------------------------------

## Rank-based diagnostics

In [34]:
mcmc_rank_hist(draws)

------------------------------------------------------------------------

### Rank-based diagnostics

Rank ECDF differences should fluctuate around zero without systematic separation between chains.

In [35]:
mcmc_rank_ecdf(draws, prob = 0.99)

------------------------------------------------------------------------

## Trace plots

In [36]:
mcmc_trace(draws)

------------------------------------------------------------------------

## Autocorrelation plots

In [37]:
mcmc_acf(draws)

------------------------------------------------------------------------

## Acceptance rates

The acceptance rate is the proportion of proposed values accepted by the Metropolis-Hastings algorithm.

In [38]:
acceptance_rates <- c(
  mh1$acceptance_rate,
  mh2$acceptance_rate,
  mh3$acceptance_rate,
  mh4$acceptance_rate
)

acceptance_rates
mean(acceptance_rates)

------------------------------------------------------------------------

Acceptance rate is primarily a **proposal-tuning diagnostic**, not a convergence diagnostic.

For an independence sampler, efficiency depends strongly on how closely the proposal distribution resembles the target distribution and whether it adequately covers the target’s tails.

------------------------------------------------------------------------

## Comparing proposal distributions

We can investigate the effect of the proposal distribution by repeating the simulation with different Beta proposals.

For example, compare

- $\text{Beta}(1,1)$
- $\text{Beta}(2,2)$
- $\text{Beta}(3,3)$
- $\text{Beta}(44,54)$

------------------------------------------------------------------------

The first three proposals are relatively diffuse compared with the posterior distribution.

The $\text{Beta}(44,54)$ proposal is much closer to the posterior.

In [39]:
# Compare a proposal closer to the target
set.seed(78)
jpar <- c(44, 54)
mh1_new <- mh(B, 0.05, jpar, chain_id = 1)
mh2_new <- mh(B, 0.25, jpar, chain_id = 2)
mh3_new <- mh(B, 0.75, jpar, chain_id = 3)
mh4_new <- mh(B, 0.95, jpar, chain_id = 4)

------------------------------------------------------------------------

In [40]:
acceptance_rates <- c(
  mh1_new$acceptance_rate,
  mh2_new$acceptance_rate,
  mh3_new$acceptance_rate,
  mh4_new$acceptance_rate
)
acceptance_rates
mean(acceptance_rates)

------------------------------------------------------------------------

### Example: Componentwise M-H for $N(\mu,\sigma^2)$

Implement a componentwise Metropolis-Hastings sampler to draw samples from the posterior for the parameters $\mu$ and $\sigma^2$ for the midge data introduced in Chapter 3.

- **Data distribution:** $y_1,\ldots,y_n \mid \mu,\sigma^2 \stackrel{\text{i.i.d.}}{\sim} N(\mu,\sigma^2)$.
- **Prior distributions:**
  - $\mu \mid \sigma^2 \sim N(\mu_0,\sigma^2/\kappa_0)$ with $\mu_0=1.9$ and $\kappa_0=1$.
  - $\sigma^2 \sim \text{Inv-}\chi^2_{\nu_0}(\sigma_0^2)$ with $\nu_0=1$ and $\sigma_0^2=0.1^2$.
- **Proposal distributions:**
  - $\mu^* \mid \mu^{(t-1)} \sim N(\mu^{(t-1)},\sigma_\mu^2)$ with $\sigma_\mu^2=0.2^2$.
  - $(\sigma^2)^* \sim \text{Inv-}\chi^2_{\nu_*}(s_*^2)$ with $\nu_*=2$ and $s_*^2=0.1^2$.

------------------------------------------------------------------------

## Acceptance ratios for componentwise Metropolis-Hastings

The joint posterior density is

$$
p(\mu,\sigma^2 \mid y)
\propto
p(y \mid \mu,\sigma^2)
p_{\mu}(\mu \mid \sigma^2)
p_{\sigma^2}(\sigma^2).
$$

We update $\mu$ and $\sigma^2$ sequentially.

------------------------------------------------------------------------

## Updating $\mu$

We use a symmetric normal random-walk proposal:

$$
\mu^* \mid \mu^{(t-1)}
\sim
N\left(\mu^{(t-1)},\sigma^2_{\mu}\right).
$$

T he proposal densities cancel because the proposal distribution is symmetric ($j_{t,\mu}(\mu^* \mid \mu^{(t-1)}) = j_{t,\mu}(\mu^{(t-1)} \mid \mu^*)$).

Additionally, the priors for $\sigma^2$ will also cancel since we will evaluate both at $\sigma^{2(t-1)}$.

------------------------------------------------------------------------

The acceptance ratio simplifies to

$$
r_\mu
=
\frac{
p(y \mid \mu^*,\sigma^{2(t-1)})
p_\mu(\mu^* \mid \sigma^{2(t-1)})
}{
p(y \mid \mu^{(t-1)},\sigma^{2(t-1)})
p_\mu(\mu^{(t-1)} \mid \sigma^{2(t-1)})
}.
$$

------------------------------------------------------------------------

## Updating $\sigma^2$

We use an $\text{Inv-}\chi^2$ independence proposal:

$$
(\sigma^2)^* \sim \text{Inv-}\chi^2_{\nu_*}(s_*^2)
$$ with $\nu_*=2$ and $s_*^2=0.1^2$.

Let $j(\sigma^2)$ denote the proposal density. The acceptance ratio is

$$
r_{\sigma^2}
=
\frac{
p(y \mid \mu^{(t)},(\sigma^{2})^*)
p_\mu(\mu^{(t)} \mid (\sigma^{2})^*)
p_{\sigma^2}((\sigma^{2})^*)
j(\sigma^{2(t-1)})
}{
p(y \mid \mu^{(t)},\sigma^{2(t-1)})
p_\mu(\mu^{(t)} \mid \sigma^{2(t-1)})
p_{\sigma^2}(\sigma^{2(t-1)})
j((\sigma^{2})^*)
}.
$$

Notice that the prior density $p_\mu(\mu \mid \sigma^2)$ does **not** cancel, because it depends on $\sigma^2$, even though $\mu$ is held fixed.

------------------------------------------------------------------------

In [41]:
# problem setup
y <- c(1.64, 1.70, 1.72, 1.74, 1.82, 1.82, 1.82, 1.90, 2.08)
n <- length(y)
mu0 <- 1.9
k0 <- 1
nu0 <- 1
sigmasq0 <- 0.01
B <- 100000
m <- mean(y)
s <- sd(y)
scale_sigmasq <- 0.01

In [42]:
log_posterior <- function(mu, sigmasq) {
  sum(dnorm(y, mean = mu, sd = sqrt(sigmasq), log = TRUE)) +
    dnorm(mu, mean = mu0, sd = sqrt(sigmasq / k0), log = TRUE) +
    bayesutils::dinvchisq(sigmasq, df = nu0, scale = sigmasq0, log = TRUE)
}

------------------------------------------------------------------------

### Componentwise M-H sampler

In [43]:
cmh <- function(B, start, jump_parm1, jump_parm2, chain_id = runif(1)) {
  mustar_sigma <- jump_parm1
  df_sigmasqstar <- jump_parm2[1]
  scale_sigmasqstar <- jump_parm2[2]

  theta <- matrix(0, nrow = B + 1, ncol = 2)
  theta[1, ] <- start

  mu_current <- start[1]
  sigmasq_current <- start[2]

  accepted_mu <- 0
  accepted_sigmasq <- 0

  for (i in 2:(B + 1)) {
    # Update mu using a symmetric normal random-walk proposal
    mu_star <- rnorm(1, mean = mu_current, sd = mustar_sigma)

    logr <- log_posterior(mu_star, sigmasq_current) -
      log_posterior(mu_current, sigmasq_current)

    if (log(runif(1)) <= min(logr, 0)) {
      mu_current <- mu_star
      accepted_mu <- accepted_mu + 1
    }

    # Update sigmasq using an inverse-chi-squared independence proposal
    sigmasq_star <- bayesutils::rinvchisq(
      1, df = df_sigmasqstar, scale = scale_sigmasqstar
    )

    logr <- log_posterior(mu_current, sigmasq_star) -
      log_posterior(mu_current, sigmasq_current) +
      dinvchisq(sigmasq_current, df = df_sigmasqstar,
                scale = scale_sigmasqstar, log = TRUE) -
      dinvchisq(sigmasq_star, df = df_sigmasqstar,
                scale = scale_sigmasqstar, log = TRUE)

    if (log(runif(1)) <= min(logr, 0)) {
      sigmasq_current <- sigmasq_star
      accepted_sigmasq <- accepted_sigmasq + 1
    }

    # Store updated parameters
    theta[i, ] <- c(mu_current, sigmasq_current)
  }

  list(
    draws = posterior::as_draws_df(
      data.frame(.chain = chain_id, .iteration = seq_len(B + 1),
                 mu = theta[, 1], sigmasq = theta[, 2])),
    acceptance_rate = c(mu = accepted_mu / B,
                        sigmasq = accepted_sigmasq / B))
}

------------------------------------------------------------------------

In [44]:
set.seed(77)

mh1 <- cmh(B, start = c(m, s^2), jump_parm1 = 0.2, 
           jump_parm2 = c(2, scale_sigmasq), chain_id = 1)
mh2 <- cmh(B, start = c(1, 0.5^2), jump_parm1 = 0.2,
           jump_parm2 = c(2, scale_sigmasq), chain_id = 2)
mh3 <- cmh(B, start = c(1, 0.5^2), jump_parm1 = 0.2,
           jump_parm2 = c(2, scale_sigmasq), chain_id = 3)
mh4 <- cmh(B, start = c(1, 0.5^2), jump_parm1 = 0.2,
           jump_parm2 = c(2, scale_sigmasq), chain_id = 4)

In [45]:
draws_all <- bind_draws(mh1$draws, mh2$draws, mh3$draws, mh4$draws, 
                        along = "chain")
# Discard the initial state and first half of transitions
draws <- subset_draws(draws_all, iteration = (B / 2 + 2):(B + 1))

------------------------------------------------------------------------

## Numerical diagnostics

In [46]:
# R-hat, bulk ESS, tail ESS, and MCSE
summarize_draws(draws)

------------------------------------------------------------------------

## Rank histograms

In [47]:
mcmc_rank_hist(draws)

------------------------------------------------------------------------

## Rank ECDFs

In [48]:
# computationally expensive
# mcmc_rank_ecdf(draws, K = 100, prob = 0.99)

------------------------------------------------------------------------

## Trace plots

In [49]:
mcmc_trace(draws)

------------------------------------------------------------------------

## Autocorrelation plots

In [50]:
mcmc_acf(draws)

------------------------------------------------------------------------

## Posterior density overlays

In [51]:
mcmc_dens_overlay(draws)

------------------------------------------------------------------------

## Joint posterior density plot

In [52]:
#mcmc_scatter(draws, pars = c("mu", "sigmasq"))
mcmc_hex(draws, pars = c("mu", "sigmasq"), bins = 30)

------------------------------------------------------------------------

## Acceptance rates

In [53]:
acceptance_rates <- rbind(mh1$acceptance_rate, mh2$acceptance_rate,
  mh3$acceptance_rate, mh4$acceptance_rate)
rownames(acceptance_rates) <- paste0("Chain ", 1:4)
acceptance_rates

In [54]:
# mean acceptance rates
colMeans(acceptance_rates)

------------------------------------------------------------------------

## Numeric posterior summary

In [55]:
summarize_draws(draws, mean, median, sd, quantile2)

------------------------------------------------------------------------

## Example: M-H independence chain for mixture distribution

Implement an independence chain Metropolis-Hastings sampler to draw samples from the posterior distribution of $\theta$ using a sample coming from a mixture distribution.

- **Data distribution:** $y_1,\ldots,y_n \mid \theta \stackrel{\text{i.i.d.}}{\sim} p(y_i \mid \theta)$, with $$
  p(y_i \mid \theta) = \theta N(7,0.5^2) + (1-\theta)N(10,0.5^2).
  $$
- **Prior distribution:** $\theta \sim \text{Beta}(1,1)$.
- **Proposal distribution:** $\theta^* \sim \text{Beta}(\alpha,\beta)$.

------------------------------------------------------------------------

In [56]:
y <- c(
  7.25325222659913, 6.85652267046824, 7.23643792894966,
  7.03343611519664, 6.9186591609056, 6.65649879051228,
  6.42308043084932, 7.46636287619574, 10.3497865413661,
  6.93593298389149, 6.83974994639286, 10.1477534866707,
  7.18844547660898, 8.79161716373787, 6.77135115622428,
  9.89206349173715, 10.6292620609587, 6.17109362928208,
  9.44878709751433, 7.12422462946795, 6.75066335182976,
  7.42808832040163, 9.4949511197615, 6.74956775652862,
  9.46445384762244, 7.27348041082583, 6.98896265672564,
  7.26262394415349, 6.94244760575449, 7.28846831817204,
  9.70904705672207, 10.9878054216487, 7.45111574465272,
  6.97036693452533, 6.53291089305878, 6.52220443343591,
  6.10163473472885, 10.2820394025033, 8.13866685075031,
  9.51099560173583, 7.74154300863383, 6.14372115300404,
  6.68548657458669, 10.8689484723994, 6.73064827757487,
  10.5866677031468, 9.56384573435206, 6.99562383496413,
  6.18576529608999, 10.5254577115642, 9.44970647261562,
  9.84118730329914, 7.21312721539275, 7.83136245649827,
  10.2825737379552, 6.36363038852991, 6.74285813989089,
  6.98035358880607, 6.92964132433787, 6.84202550407557,
  7.38016635912841, 9.78362588855716, 7.58508950152404,
  9.50912675753626, 6.55132388271122, 6.88852617303433,
  7.90209596243866, 7.01301915336949, 9.93871470060288,
  7.51086841627442, 6.67441840075985, 6.10225392630878,
  7.40059858311095, 7.30520952023732, 7.21041994910681,
  10.519655486735, 10.1288125396083, 7.04575576918983,
  9.8008750675778, 10.3528228749625, 7.25796001419125,
  7.67456458165231, 6.52507043820612, 6.57628671699353,
  6.52670898857082, 7.44576437202348, 10.5860658476161,
  6.97650854570433, 6.89904378626167, 7.48642531607213,
  6.39227547871176, 7.32020055238666, 9.64975498620833,
  6.92112490660115, 10.4213997762887, 6.3455538999162,
  6.99875637216512, 6.98618105048244, 5.96048736574766,
  10.2852677946904
)

------------------------------------------------------------------------

## Likelihood and log posterior

In [57]:
# Mixture density for a single observation
lik <- function(y, theta) {
  theta * dnorm(y, mean = 7, sd = 0.5) +
    (1 - theta) * dnorm(y, mean = 10, sd = 0.5)
}

In [58]:
# Log joint posterior, up to a constant
# The Beta(1,1) prior is uniform.
log_posterior <- function(theta) {
  if (theta < 0 || theta > 1) return(-Inf)
  sum(log(lik(y, theta)))
}

------------------------------------------------------------------------

In [59]:
hist(y, breaks = 25, freq = FALSE,
     xlab = "Wing length", main = "Mixture data")
x <- seq(5, 12, length.out = 1000)
lines(x, lik(x, 0.7), col = "blue", lwd = 2)

------------------------------------------------------------------------

In [60]:
x <- seq(0, 1, length.out = 1001)
# Evaluate the log posterior on a grid
logpost <- sapply(x, log_posterior)
# Normalize numerically for plotting
post <- exp(logpost - max(logpost))
post <- post / sum(post * (x[2] - x[1]))
plot(x, post, type = "l", col = "orange", lwd = 2,
     xlab = expression(theta), ylab = "Density",
     main = "Posterior and proposal distributions")
lines(x, dbeta(x, 1, 1), col = "blue", lwd = 2)
lines(x, dbeta(x, 2, 10), col = "black", lwd = 2)
lines(x, dbeta(x, 35, 15), col = "grey", lwd = 2)
legend("topright",
       legend = c("Posterior", "Beta(1,1)", "Beta(2,10)", "Beta(35, 15)"),
      col = c("orange", "blue", "black", "grey"),
      lty = 1, lwd = 2)

------------------------------------------------------------------------

------------------------------------------------------------------------

In [62]:
independence_chain <- function(B, start, jump_parm, chain_id) {

  alpha <- jump_parm[1]
  beta <- jump_parm[2]

  theta <- numeric(B + 1)
  theta[1] <- start

  accepted <- 0

  for (i in 2:(B + 1)) {

    # Generate proposal independently of the current state
    theta_star <- rbeta(1, alpha, beta)

    # Log acceptance ratio
    logr <- log_posterior(theta_star) -
      log_posterior(theta[i - 1]) +
      dbeta(theta[i - 1], alpha, beta, log = TRUE) -
      dbeta(theta_star, alpha, beta, log = TRUE)

    # Accept or reject
    if (log(runif(1)) <= min(logr, 0)) {
      theta[i] <- theta_star
      accepted <- accepted + 1
    } else {
      theta[i] <- theta[i - 1]
    }
  }

  list(
    draws = posterior::as_draws_df(
      data.frame(
        .chain = chain_id,
        .iteration = seq_len(B + 1),
        theta = theta
      )
    ),
    acceptance_rate = accepted / B
  )
}

------------------------------------------------------------------------

In [63]:
set.seed(72)

B <- 10000
jpar <- c(1, 1)

mh1 <- independence_chain(B, 0.1, jpar, chain_id = 1)
mh2 <- independence_chain(B, 0.3, jpar, chain_id = 2)
mh3 <- independence_chain(B, 0.5, jpar, chain_id = 3)
mh4 <- independence_chain(B, 0.7, jpar, chain_id = 4)
mh5 <- independence_chain(B, 0.9, jpar, chain_id = 5)

------------------------------------------------------------------------

In [64]:
# Combine chains
draws_all <- bind_draws(mh1$draws, mh2$draws, mh3$draws,
                        mh4$draws, mh5$draws, along = "chain")

# Discard first half of iterations
draws <- subset_draws(draws_all, iteration = (B / 2 + 2):(B + 1))

------------------------------------------------------------------------

In [65]:
summarize_draws(draws)

------------------------------------------------------------------------

In [66]:
mcmc_rank_hist(draws)

------------------------------------------------------------------------

In [67]:
# computationally expensive
mcmc_rank_ecdf(draws, K = 100, prob = 0.99)

------------------------------------------------------------------------

In [68]:
mcmc_trace(draws)

------------------------------------------------------------------------

In [69]:
mcmc_acf(draws)

------------------------------------------------------------------------

In [70]:
mcmc_dens_overlay(draws)

------------------------------------------------------------------------

In [71]:
acceptance_rates <- c(
  mh1$acceptance_rate,
  mh2$acceptance_rate,
  mh3$acceptance_rate,
  mh4$acceptance_rate,
  mh5$acceptance_rate
)

acceptance_rates
mean(acceptance_rates)

------------------------------------------------------------------------

## Repeat using a Beta(2,10) independence proposal

In [72]:
set.seed(73)
jpar <- c(2, 10)
mh1_new <- independence_chain(B, 0.1, jpar, chain_id = 1)
mh2_new <- independence_chain(B, 0.3, jpar, chain_id = 2)
mh3_new <- independence_chain(B, 0.5, jpar, chain_id = 3)
mh4_new <- independence_chain(B, 0.7, jpar, chain_id = 4)
mh5_new <- independence_chain(B, 0.9, jpar, chain_id = 5)

------------------------------------------------------------------------

In [73]:
draws_new_all <- bind_draws(mh1_new$draws, mh2_new$draws, mh3_new$draws,
                            mh4_new$draws, mh5_new$draws,
  along = "chain")
draws_new <- subset_draws(draws_new_all, iteration = (B / 2 + 2):(B + 1))

# Beta(2,10) proposal
acceptance_rates_new <- c(
  mh1_new$acceptance_rate,
  mh2_new$acceptance_rate,
  mh3_new$acceptance_rate,
  mh4_new$acceptance_rate,
  mh5_new$acceptance_rate
)

------------------------------------------------------------------------

## Repeat using a Beta(35,15) independence proposal

In [74]:
set.seed(78)
jpar <- c(35, 15)
mh1_c <- independence_chain(B, 0.1, jpar, chain_id = 1)
mh2_c <- independence_chain(B, 0.3, jpar, chain_id = 2)
mh3_c <- independence_chain(B, 0.5, jpar, chain_id = 3)
mh4_c <- independence_chain(B, 0.7, jpar, chain_id = 4)
mh5_c <- independence_chain(B, 0.9, jpar, chain_id = 5)

------------------------------------------------------------------------

In [75]:
draws_c_all <- bind_draws(mh1_c$draws, mh2_c$draws, mh3_c$draws,
                            mh4_c$draws, mh5_c$draws,
  along = "chain")
draws_c <- subset_draws(draws_c_all, iteration = (B / 2 + 2):(B + 1))

# Beta(35,15) proposal
acceptance_rates_c <- c(
  mh1_c$acceptance_rate,
  mh2_c$acceptance_rate,
  mh3_c$acceptance_rate,
  mh4_c$acceptance_rate,
  mh5_c$acceptance_rate
)

------------------------------------------------------------------------

In [76]:
# Beta(1,1) proposal
summarize_draws(draws)

In [77]:
# Beta(2,10) proposal
summarize_draws(draws_new)

In [78]:
# Beta(35,15) proposal
summarize_draws(draws_c)

------------------------------------------------------------------------

## Compare acceptance rates

In [79]:
# Beta(1,1) proposal
acceptance_rates
mean(acceptance_rates)

In [80]:
# Beta(2, 10) proposal
acceptance_rates_new
mean(acceptance_rates_new)

In [81]:
# Beta(35, 15) proposal
acceptance_rates_c
mean(acceptance_rates_c)

------------------------------------------------------------------------

## Trace plot from Beta(1, 1) proposal distribution

In [82]:
mcmc_trace(draws)

------------------------------------------------------------------------

## Trace plot from Beta(2, 10) proposal distribution

In [83]:
mcmc_trace(draws_new)

------------------------------------------------------------------------

## Trace plot from Beta(35, 15) proposal distribution

In [84]:
mcmc_trace(draws_c)

------------------------------------------------------------------------

## Rank plots from Beta(1, 1) proposal

In [85]:
mcmc_rank_hist(draws)

------------------------------------------------------------------------

## Rank plots from Beta(2, 10) proposal

In [86]:
mcmc_rank_hist(draws_new)

------------------------------------------------------------------------

## Rank plots from Beta(35, 15) proposal

In [87]:
mcmc_rank_hist(draws_c)

------------------------------------------------------------------------

## ACF plots from Beta(1, 1) proposal

In [88]:
mcmc_acf(draws)

------------------------------------------------------------------------

## ACF plots from Beta(2, 10) proposal

In [89]:
mcmc_acf(draws_new)

------------------------------------------------------------------------

## ACF plots from Beta(35, 15) proposal

In [90]:
mcmc_acf(draws_c)